# Feasibility of EODAG

In [74]:
from pathlib import Path

import json
import pystac_client
from eodag import EODataAccessGateway

import utils.harvester_code as harvester

dag = EODataAccessGateway()

download_path = f"{Path.home()}/Daten/harvester_eodag"


### Copernicus Dataspace Ecosystem

#### 1. Searching Data with OData API

In [62]:
#url = "https://datahub.creodias.eu/odata/v1"
url = "https://catalogue.dataspace.copernicus.eu/odata/v1"
limit: int = 1000
start_time = "2026-04-29T15:00:00.000Z"
end_time = "2026-04-29T16:00:00.000Z"

filter_base = f"(PublicationDate ge {start_time} and PublicationDate lt {end_time}) and Online eq true"

#filter_s1_slc = "(startswith(Name,'S1') and contains(Name,'SLC') and not contains(Name,'_COG') and not contains(Name, 'CARD_BS'))"
#filter_s1_grd = "(startswith(Name,'S1') and contains(Name,'GRD') and not contains(Name,'_COG') and not contains(Name, 'CARD_BS'))"
filter_s2_l2a = "(startswith(Name,'S2') and (contains(Name,'L2A')) and not contains(Name,'_N9999') and not contains(Name,'_N0500'))"
#filter_s2_l1c = "(startswith(Name,'S2') and (contains(Name,'L1C')) and not contains(Name,'_N9999') and not contains(Name,'_N0500'))"
#filter_s3 = "(startswith(Name,'S3') and not contains(Name,'NRTI_'))"
#filter_s5p = "(startswith(Name,'S5P') and not contains(Name,'NRTI_'))"

filters = [f"({filter_base}) and ({filter_s2_l2a})"]

odata_scenes = harvester.search(
    api_url=url,
    max_items=limit,
    filters=filters,
)

Executing OData query: https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=((PublicationDate ge 2026-04-29T15:00:00.000Z and PublicationDate lt 2026-04-29T16:00:00.000Z) and Online eq true) and ((startswith(Name,'S2') and (contains(Name,'L2A')) and not contains(Name,'_N9999') and not contains(Name,'_N0500')))&$top=1000
Found 654 scenes
Result set complete


#### 2. Searching Data with eodag
We cannot use the `start` and `end` parameters of the `search()` function since we want to filter by publication time instead of sensing time.

In [69]:
eodag_products = dag.search(
    provider="cop_dataspace",
    collection="S2_MSI_L2A",
    #start=start_time,
    #end=end_time,
    #geom="POLYGON((3 55, 3 47, 18 47, 18 55, 3 55))",
    published_after=start_time,
    published_before=end_time,
    limit=limit
)
print(f"Found {len(eodag_products.data)} products")

Found 654 products


#### 3. Compare search results
Check for each OData result if eodag has found the same product.

In [64]:
matching_product_uid = []
for odata_scene in odata_scenes:
    uid = odata_scene["uid"]
    for eodag_product in eodag_products:
        if eodag_product.properties["uid"] == uid:
            matching_product_uid.append(uid)            

print(f"{len(odata_scenes)} OData products found")
print(f"{len(eodag_products)} eodag products found")
print(f"{len(matching_product_uid)} Matching products")

654 OData products found
654 eodag products found
654 Matching products


#### 4. Downloading data with eodag

In [70]:
paths = dag.download(
    eodag_products[1],
    output_dir=download_path,
    extract=False
)

0.00B [00:00, ?B/s]

### USGS

#### 1. Searching Data at STAC API

In [6]:
usgs_stac_url = "https://landsatlook.usgs.gov/stac-server"
collection = "landsat-c2l2-sr"
limit = 1000
start_time = "2026-05-04T00:00:00.000Z"
#end_time = "2026-05-04T17:00:00.000Z"         #   <-- at this time not all data ingested into stac catalog but m2m
#end_time = "2026-05-04T11:00:00.000Z"          #   <-- 12pm vs. 10am = 77 results
end_time = "2026-05-04T23:59:59.000Z"

bbox = None
stacql_query = {
    "created": {"gte": start_time, "lt": end_time},
}

print(f"Collection: {collection}")
print(f"BBox: {bbox}")
print(f"STAC Query: {json.dumps(stacql_query, indent=4)}")
print(f"Limit: {limit}\n")

client = pystac_client.Client.open(usgs_stac_url)
usgs_items = client.search(
    collections=[collection],
    bbox=bbox,
    datetime=None,
    max_items=limit,
    query=stacql_query
).item_collection()

print(f"Found {len(usgs_items)} products")

Collection: landsat-c2l2-sr
BBox: None
STAC Query: {
    "created": {
        "gte": "2026-05-04T00:00:00.000Z",
        "lt": "2026-05-04T23:59:59.000Z"
    }
}
Limit: 2000

Found 753 products


In [7]:
usgs_items[0]

<Item id=LC09_L2SP_095022_20260503_20260504_02_T2_SR>

#### 2. Searching Data with eodag
We cannot use the `start` and `end` parameters of the `search()` function since we want to filter by publication time instead of sensing time.

**Challenge**
- Data sets differ between STAC(not sensor-based. uses level and sr/st(?) for collections) and M2M(collections are sensor-based)
- There is no plugin for USGS data that allows users to perform an EODAG search via STAC and download data via m2m (as was the case with the DLR approach). And there is no "created" date parameter or anything similar supported via m2m API Plugin from EODAG Search.

**Solution**
1) The collection problem can be solved by using the "landsat_ot_c2_l2" collection. This collection contains the data from the two ongoing Landsat 8 and 9 missions.
2) C-Scale implemented an ingestion Filter Option (ingestFilter in scene-search Endpoint of m2m API). This enables a "filter by publication" kind of search. The timestamps do not match, but ingestionFilter search is functionally equivalent to the publishDate filter. So there is no need for a STAC+m2m approach anymore. Everything runs over m2m.

In [50]:

ingest_filter = {"start": start_time, "end": end_time}

scene_filter_input = {
    "ingestFilter": ingest_filter
}


usgs_products = dag.search(
    provider="usgs",
    # "TBD: eodag type for landsat-c2l2-sr",
    collection="landsat_ot_c2_l2",
    bbox=bbox,
    #start="2018-06-01",
    #end="2019-06-15",
    limit=limit,
    #TBD: find the param representing the 'created' timestamp   
    scene_filter=scene_filter_input,
)

print(f"Found {len(usgs_products.data)} products")

Found 753 products


#### 3. Compare search results
Check for each USGS STAC result if EODAG has found the same product.

In [51]:
a = {i.properties["id"] for i in usgs_products.data}
b = {i.id.replace("_SR", "") for i in usgs_items}

c = a ^ b    # all mismatched Elements 
d = a & b    # all matching Elements

print(f"Number of mismatched items: {len(c)}")
print(f"Number of matching items: {len(d)}")


Number of mismatched items: 0
Number of matching items: 753


In [52]:
mapping_usgs = {i.properties["id"]: i for i in usgs_products.data}
mapping_items = {i.id.replace("_SR", ""): i for i in usgs_items}

matching_ids = set(mapping_usgs.keys()) & set(mapping_items.keys())

matching_pairs = []
for matching_id in matching_ids:
    obj_a = mapping_usgs[matching_id]
    obj_b = mapping_items[matching_id]
    matching_pairs.append((obj_a, obj_b))

print(f"Matching Pairs Count: {len(matching_pairs)}")


Matching Pairs Count: 753


In [53]:
chosen_id = None
for i, pair in enumerate(matching_pairs):
    if 'LC09_L2SP_177045_20260502_20260504_02_T1' in pair[0].properties["id"]:
        chosen_id = i

matching_pairs[chosen_id][0]

EOProduct(id=LC09_L2SP_177045_20260502_20260504_02_T1, provider=usgs)

In [54]:
matching_pairs[chosen_id][1]

<Item id=LC09_L2SP_177045_20260502_20260504_02_T1_SR>

**IDs of chosen matching objects**
- 'LC09_L2SP_177045_20260502_20260504_02_T1' - m2m API/EODAG Product
- "LC09_L2SP_177045_20260502_20260504_02_T1_SR" - STAC Item

**Timestamps (published vs. created)**
- '2026-05-04T10:21:49.000Z' - "published"-EODAG property
- "2026-05-04T10:55:33.468Z" - "created"-STAC Property

#### 4. Download the Data

In [71]:
paths = dag.download(
    matching_pairs[chosen_id][0],
    output_dir=download_path,
    extract=False
)

0.00B [00:00, ?B/s]